
# **Lab 6: Improving Models with Feature Engineering**
**Unit 2 • Week 9 (Thu) — Feature Engineering & Advanced Models**

**Objective:** Improve your **Lab 5** classifier by creating **new features** using the BQML `TRANSFORM` clause, then compare performance to a **baseline**.


## Setup & Authentication

In [18]:

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "big-data-analysis-472319"
FULL_TABLE = "bigquery-samples.airline_ontime_data.flights"  # update if needed

BASELINE_MODEL = f"{PROJECT_ID}.superstore_data.flight_diverted_classifier"  # from Lab 5
IMPROVED_MODEL = f"{PROJECT_ID}.superstore_data.flight_diverted_classifier_v2"

client = bigquery.Client(project=PROJECT_ID)
print("Authenticated. Project:", PROJECT_ID)


Authenticated. Project: big-data-analysis-472319



---
## Establish a Baseline — Validate

Re-run `ML.EVALUATE` for your **Lab 5** model and record key metrics.


In [19]:

baseline_sql = f"SELECT * FROM ML.EVALUATE(MODEL `{BASELINE_MODEL}`)"
baseline_df = client.query(baseline_sql).result().to_dataframe()
baseline_df


,precision,recall,accuracy,f1_score,log_loss,roc_auc
0,0.899796,0.810313,0.984611,0.852713,0.051433,0.9784



---
## Brainstorm New Features — Investigate (Gemini Prompt)

```python
prompt =
# TASK: Brainstorm new features for an ML model, a process called feature engineering.
# CONTEXT: I want to improve my flight diversion prediction model. The raw data has a column called 'departure_airport' (e.g., 'JFK', 'ORD') and another called 'airline' (e.g., 'AA', 'UA').
# GOAL: Suggest one new feature I could create by combining 'departure_airport' and 'airline' that might be more predictive than either column alone. Explain why this new feature could be more powerful.
```



---
## Implement Feature Engineering with `TRANSFORM`

Modify your `CREATE MODEL` from Lab 5 and add a `TRANSFORM` clause to create engineered features.

Requested features:
1. **`route`** = CONCAT(`origin`, '-', `dest`)  
2. **`day_of_week`** = EXTRACT(DAYOFWEEK FROM `fl_date`)


In [38]:
create_v2_sql = f"""
CREATE OR REPLACE MODEL `{IMPROVED_MODEL}`
TRANSFORM (
  -- pass-through columns
  *,
  -- engineered features
  CONCAT(CAST(departure_airport AS STRING), '-', CAST(arrival_airport AS STRING)) AS route,
  EXTRACT(DAYOFWEEK FROM CAST(date AS DATE)) AS day_of_week
)
OPTIONS(
  model_type='logistic_reg',
  input_label_cols=['is_diverted'],
  enable_global_explain=TRUE
)
AS
SELECT
  CASE WHEN arrival_delay > 60 THEN 1 ELSE 0 END AS is_diverted,
  departure_delay,
  departure_airport,
  arrival_airport,
  date,
  arrival_delay
FROM `{FULL_TABLE}`
WHERE arrival_delay IS NOT NULL
  AND departure_delay IS NOT NULL
LIMIT 500000;
"""

job = client.query(create_v2_sql)
job.result()
print("Improved model created:", IMPROVED_MODEL)

Improved model created: big-data-analysis-472319.superstore_data.flight_diverted_classifier_v2



---
## Compare Performance — Extend

Evaluate the improved model and compare against the baseline.


In [39]:
improved_sql = f"SELECT * FROM ML.EVALUATE(MODEL `{IMPROVED_MODEL}`)"
improved_df = client.query(improved_sql).result().to_dataframe()
improved_df

,precision,recall,accuracy,f1_score,log_loss,roc_auc
0,0.957115,0.910946,0.99285,0.93346,0.021192,0.999848



Create a small comparison table below in Markdown or code (precision, recall, etc.):
- **Baseline (Lab 5)**: The baseline model used only basic features like departure and arrival delays, providing solid accuracy but limited predictive depth. While performance was strong (accuracy 0.9846, ROC AUC 0.9784), the model lacked contextual understanding of flight patterns.  
- **Improved (Lab 6)**: After adding engineered features such as route and day_of_week, the model’s precision, recall, and F1 score increased significantly. This shows that feature engineering helped the model better capture flight behavior and improved its ability to predict diversions more accurately and confidently.

**Did feature engineering improve performance? Why or why not?**

Yes, feature engineering significantly improved performance.
By creating new features like route and day_of_week, the model captured contextual travel patterns that better explain flight delays. This led to higher precision, recall, and F1 scores, showing the model became more accurate and confident in predicting diversions.


---
## Challenge: `ML.BUCKETIZE`

Author your own Gemini prompt to write a `TRANSFORM` clause that buckets `dep_delay` into 4 severity levels (e.g., early/on-time, minor, moderate, major).

> Hint: `ML.BUCKETIZE(dep_delay, [boundary_list])` returns a **bucket index**; you can also map buckets with `CASE`.

prompt =
Act as a senior BigQuery ML analyst. Write a TRANSFORM clause for a BigQuery ML CREATE MODEL statement that creates a new feature called dep_delay_bucket, which categorizes departure_delay into four severity levels: Early/On-Time Minor Delay Moderate Delay Major Delay Use either ML.BUCKETIZE(departure_delay, [boundary_list]) or a CASE expression to assign each trip to one of these categories based on delay thresholds (for example: 0, 15, 30, 60 minutes). Ensure the TRANSFORM clause is properly formatted, runnable in BigQuery, and includes both the original departure_delay and the new bucketed feature.


In [40]:
transform_clause = """
TRANSFORM (
  -- Include the original departure_delay
  CAST(departure_delay AS FLOAT64) AS departure_delay,
  -- Create the new dep_delay_bucket feature
  CASE
    WHEN departure_delay <= 0 THEN 'Early/On-Time'
    WHEN departure_delay > 0 AND departure_delay <= 15 THEN 'Minor Delay'
    WHEN departure_delay > 15 AND departure_delay <= 60 THEN 'Moderate Delay'
    WHEN departure_delay > 60 THEN 'Major Delay'
    ELSE 'Unknown' -- Handle potential NULL or unexpected values
  END AS dep_delay_bucket
  -- You would include other features and the label here as well
  -- e.g., CONCAT(CAST(departure_airport AS STRING), '-', CAST(arrival_airport AS STRING)) AS route,
  --       EXTRACT(DAYOFWEEK FROM CAST(date AS DATE)) AS day_of_week,
  --       CASE WHEN arrival_delay > 60 THEN 1 ELSE 0 END AS is_diverted
)
"""

print(transform_clause)


TRANSFORM (
  -- Include the original departure_delay
  CAST(departure_delay AS FLOAT64) AS departure_delay,
  -- Create the new dep_delay_bucket feature
  CASE
    WHEN departure_delay <= 0 THEN 'Early/On-Time'
    WHEN departure_delay > 0 AND departure_delay <= 15 THEN 'Minor Delay'
    WHEN departure_delay > 15 AND departure_delay <= 60 THEN 'Moderate Delay'
    WHEN departure_delay > 60 THEN 'Major Delay'
    ELSE 'Unknown' -- Handle potential NULL or unexpected values
  END AS dep_delay_bucket
  -- You would include other features and the label here as well
  -- e.g., CONCAT(CAST(departure_airport AS STRING), '-', CAST(arrival_airport AS STRING)) AS route,
  --       EXTRACT(DAYOFWEEK FROM CAST(date AS DATE)) AS day_of_week,
  --       CASE WHEN arrival_delay > 60 THEN 1 ELSE 0 END AS is_diverted
)



In [43]:
select_transformed_sql = f"""
SELECT
  departure_delay,
  CASE
    WHEN departure_delay <= 0 THEN 'Early/On-Time'
    WHEN departure_delay > 0 AND departure_delay <= 15 THEN 'Minor Delay'
    WHEN departure_delay > 15 AND departure_delay <= 60 THEN 'Moderate Delay'
    WHEN departure_delay > 60 THEN 'Major Delay'
    ELSE 'Unknown' -- Handle potential NULL or unexpected values
  END AS dep_delay_bucket
FROM
  `{FULL_TABLE}`
WHERE departure_delay IS NOT NULL
LIMIT 10;
"""

transformed_delay_df = client.query(select_transformed_sql).result().to_dataframe()
display(transformed_delay_df)

,departure_delay,dep_delay_bucket
0,-6.0,Early/On-Time
1,29.0,Moderate Delay
2,0.0,Early/On-Time
3,-4.0,Early/On-Time
4,-2.0,Early/On-Time
5,-6.0,Early/On-Time
6,8.0,Minor Delay
7,-2.0,Early/On-Time
8,-4.0,Early/On-Time
9,-5.0,Early/On-Time



---
## ✅ Deliverable for Lab 6

- Completed `Lab6_Feature_Engineering.ipynb` showing:
  - Baseline metrics
  - Engineered features via `TRANSFORM`
  - Improved model evaluation + comparison
- Push to **GitHub** and submit the link on **Brightspace**.
